**Navigation** : [Version Python](Z3-13b-UnsatCores-MUS-Python.ipynb) | [13 - UNSAT cores](Z3-13-UnsatCores-Python.ipynb) | [README](README.md) | [14 - Bit-vectors](Z3-14-BitVectors-Overflow-Python.ipynb)

# Z3-CSharp-13b — UNSAT cores et MUS

**Objectifs** : extraire un core et construire un MUS irréductible par suppression.  
**Prérequis** : contraintes Z3 et notebook 13. **Durée** : 40 minutes.

Un core est un sous-ensemble insatisfiable ; un MUS est minimal par inclusion, pas nécessairement de cardinal minimum.

Le binding NuGet est fixé à 4.12.2. `Z3NativeLoader` charge ici la bibliothèque native 5.4.1.0 ; les appels de ce carnet sont exécutés ensemble dans cette configuration.

## Source

Le livre-ancrage de la série est **Yurichev, Dennis — *SAT/SMT by Example*** (édition publique du 12 février 2023, 666 pages) : §3.13.2 « Unsat core » (p. 90) et §22.1 « Minimal Unsatisfiable Subformula » (p. 590, chapitre 22 « Everything else »).

Le §3.13.2 pose l'extraction d'un core côté Z3 : `assert_and_track` étiquette les contraintes, `unsat_core()` isole celles qui se contredisent — son exemple est un tableur à dépendance circulaire `C0` ↔ `C1`, dont le core vaut `[C0, C1]` ; le §3.13.3 « Stress test » (p. 92) monte la charge sur un tableur aléatoire. Le §22.1 attaque le **MUS** : à partir du problème de Knuth (TAOCP 7.1.11), l'auteur fait calculer un MUS par PicoMUS — « computed MUS of size 8 out of 16 (50 %) » (p. 592) — et rappelle qu'un MUS est le sous-ensemble dont le retrait de toute clause rend l'instance satisfiable.

Le carnet **transporte le MUS au niveau des contraintes Z3** : il montre que `unsat_core()` ne garantit pas la minimalité, puis construit le sous-ensemble irréductible par suppression successive (`AssertAndTrack` / `UnsatCore` sur `Microsoft.Z3`), sur une chaîne de contraintes dont l'insatisfiabilité n'apparaît dans aucune paire. Le jumeau Python [`Z3-13b-UnsatCores-MUS-Python.ipynb`](Z3-13b-UnsatCores-MUS-Python.ipynb) traite le même enchaînement côté `pyz3`.

Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16 456 233 octets). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.


In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Linq;
using System.Collections.Generic;

Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine($".NET {Environment.Version} ; binding Microsoft.Z3 4.12.2 ; bibliothèque native Z3 {Microsoft.Z3.Version.FullVersion}");


Installed Packages Microsoft.Z3, 4.12.2

.NET 10.0.14 ; binding Microsoft.Z3 4.12.2 ; bibliothèque native Z3 5.4.1.0


## 1. Une chaîne impossible
Quatre tâches de durée 2 se succèdent, démarrent après zéro et finissent avant 7. Les 11 contraintes sont suivies par leurs étiquettes.

In [2]:
// Chaque formule possède une étiquette distincte pour relier le core au modèle de tâches.
// La durée2 sur quatre tâches force une fin à8, incompatible avec la deadline7 ; les autres bornes ne sont pas retirées du système.
var ctx = new Context();
var starts = Enumerable.Range(0, 4).Select(i => ctx.MkIntConst($"start_{i}")).ToArray();
var chain = new List<(BoolExpr formula, string label)>();
for (int i = 0; i < 4; i++)
{
    chain.Add((ctx.MkGe(starts[i], ctx.MkInt(0)), $"release_T{i}"));
    chain.Add((ctx.MkLe(ctx.MkAdd(starts[i], ctx.MkInt(2)), ctx.MkInt(7)), $"deadline_T{i}"));
}

for (int i = 0; i < 3; i++)
    chain.Add((ctx.MkLe(ctx.MkAdd(starts[i], ctx.MkInt(2)), starts[i + 1]), $"precedence_T{i}_vers_T{i + 1}"));
Solver Tracked(List<(BoolExpr formula, string label)> xs)
{
    var solver = ctx.MkSolver();
    foreach (var(formula, label)in xs)
        solver.AssertAndTrack(formula, ctx.MkBoolConst(label));
    return solver;
}

var chainSolver = Tracked(chain);
Console.WriteLine($"Statut : {chainSolver.Check()}");
var chainCore = chainSolver.UnsatCore.Select(x => x.ToString()).ToHashSet();
Console.WriteLine($"Core ({chainCore.Count}/{chain.Count}) : {string.Join(", ", chainCore.OrderBy(x => x))}");
Console.WriteLine($"Hors core : {string.Join(", ", chain.Select(x => x.label).Where(x => !chainCore.Contains(x)))}");


Statut : UNSATISFIABLE


Core (5/11) : deadline_T3, precedence_T0_vers_T1, precedence_T1_vers_T2, precedence_T2_vers_T3, release_T0


Hors core : deadline_T0, release_T1, deadline_T1, release_T2, deadline_T2, release_T3


La chaîne demande 8 unités alors que l'échéance vaut 7. Le core identifie les contraintes suffisantes ; sa minimalité ne découle pas de l'API.

### Exercice 1 — Vérifier l'irréductibilité
Dans les huit contraintes ci-dessous, fournissez les étiquettes d’un MUS. Le garde vérifie leur présence, UNSAT et SAT après chaque retrait.

In [3]:
Status Check(List<(BoolExpr formula, string label)> xs)
{
    using var s = ctx.MkSolver();
    s.Assert(xs.Select(x => x.formula).ToArray());
    return s.Check();
}

bool IsMus(List<(BoolExpr formula, string label)> all, string[] answer)
{
    if (answer == null || answer.Length == 0 || answer.Distinct().Count() != answer.Length || answer.Any(x => !all.Any(y => y.label == x)))
        return false;
    var sub = all.Where(x => answer.Contains(x.label)).ToList();
    return Check(sub) == Status.UNSATISFIABLE && sub.All(x => Check(sub.Where(y => y.label != x.label).ToList()) == Status.SATISFIABLE);
}

var w = ctx.MkIntConst("w");
var ex1Constraints = new List<(BoolExpr formula, string label)>
{
    (ctx.MkGe(w, ctx.MkInt(0)), "nonneg"),
    (ctx.MkGe(w, ctx.MkInt(4)), "lower"),
    (ctx.MkLe(ctx.MkAdd(w, ctx.MkInt(4)), ctx.MkInt(6)), "upper"),
    (ctx.MkEq(w, ctx.MkInt(5)), "equal"),
    (ctx.MkLe(w, ctx.MkInt(100)), "bound"),
    (ctx.MkNot(ctx.MkEq(w, ctx.MkInt(7))), "not7"),
    (ctx.MkGe(w, ctx.MkInt(-10)), "floor"),
    (ctx.MkLe(w, ctx.MkInt(50)), "ceiling")
};
string[] exercice1 = null; // TODO : étiquettes d'un MUS de vos huit contraintes
Console.WriteLine(exercice1 == null ? "Exercice 1 à compléter" : $"MUS valide : {IsMus(ex1Constraints, exercice1)}");


Exercice 1 à compléter


## 2. Suppression successive
On teste chaque retrait sur la liste entière ; un retrait conservant UNSAT est définitif. UNKNOWN ne prouve pas SAT et ne doit pas être traité comme tel.

In [4]:
// La suppression part de toute la listeUNSAT et ne retire que les contraintes laissantUNSAT.
// SAT rend le membre essentiel ; UNKNOWN est refusé comme preuve. La minimalité obtenue est par inclusion, sans cardinal minimum garanti.
List<(BoolExpr formula, string label)> ComputeMus(List<(BoolExpr formula, string label)> constraints)
{
    var mus = constraints.ToList();
    var initialStatus = Check(mus);
    if (initialStatus == Status.SATISFIABLE)
        throw new ArgumentException("Le système doit être UNSAT.");
    if (initialStatus == Status.UNKNOWN)
        throw new InvalidOperationException("Z3 a renvoyé UNKNOWN.");
    for (int i = 0; i < mus.Count;)
    {
        var candidate = mus.Where((x, j) => j != i).ToList();
        var status = Check(candidate);
        if (status == Status.UNSATISFIABLE)
            mus = candidate;
        else if (status == Status.SATISFIABLE)
            i++;
        else
            throw new InvalidOperationException("Z3 a renvoyé UNKNOWN.");
    }

    return mus;
}

var t = Enumerable.Range(0, 5).Select(i => ctx.MkIntConst($"t{i}")).ToArray();
var edges = new List<(BoolExpr formula, string label)>
{
    (ctx.MkGe(t[1], ctx.MkAdd(t[0], ctx.MkInt(5))), "e01"),
    (ctx.MkGe(t[2], ctx.MkAdd(t[1], ctx.MkInt(3))), "e12"),
    (ctx.MkGe(t[3], ctx.MkAdd(t[2], ctx.MkInt(4))), "e23"),
    (ctx.MkGe(t[3], ctx.MkAdd(t[0], ctx.MkInt(12))), "e03"),
    (ctx.MkGe(t[4], ctx.MkAdd(t[3], ctx.MkInt(2))), "e34"),
    (ctx.MkLe(t[4], ctx.MkAdd(t[0], ctx.MkInt(10))), "dead")
};
var raw = Tracked(edges);
Console.WriteLine($"Statut : {raw.Check()}");
var rawLabels = raw.UnsatCore.Select(x => x.ToString()).OrderBy(x => x).ToArray();
var mus = ComputeMus(edges);
var musLabels = mus.Select(x => x.label).OrderBy(x => x).ToArray();
Console.WriteLine($"Core brut ({rawLabels.Length}) : {string.Join(", ", rawLabels)}");
Console.WriteLine($"MUS ({musLabels.Length}) : {string.Join(", ", musLabels)} ; irréductible : {IsMus(edges, musLabels)}");


Statut : UNSATISFIABLE


Core brut (5) : dead, e01, e12, e23, e34


MUS (3) : dead, e03, e34 ; irréductible : True


Le chemin détaillé totalise 14 alors que l'échéance autorise 10. L'arête directe remplace les trois premières arêtes dans un autre conflit. Le core brut peut être lui aussi irréductible : sa comparaison avec un MUS différent ne prouve pas sa non-minimalité.

### Exercice 2 — Arête transitive
Sur trois tâches, le modèle ci-dessous comprend un chemin et une arête directe. Donnez les étiquettes d’un MUS et vérifiez-les avec le même garde.

Les trois arêtes détaillées `e01`, `e12`, `e23` impliquent ensemble `e03` : leurs durées s'additionnent en 12. La réciproque est fausse : savoir que `t3 >= t0 + 12` ne fixe aucune des deux dates intermédiaires. Le chemin détaillé forme donc une autre preuve, dont chaque arête peut être essentielle. Un MUS est minimal **par inclusion** : aucun de ses membres ne peut être retiré. Il n'est pas forcément de **cardinal minimum** parmi tous les conflits du système.

In [5]:
var u = Enumerable.Range(0, 3).Select(i => ctx.MkIntConst($"u{i}")).ToArray();
var ex2Constraints = new List<(BoolExpr formula, string label)>
{
    (ctx.MkGe(u[1], ctx.MkAdd(u[0], ctx.MkInt(2))), "step01"),
    (ctx.MkGe(u[2], ctx.MkAdd(u[1], ctx.MkInt(2))), "step12"),
    (ctx.MkGe(u[2], ctx.MkAdd(u[0], ctx.MkInt(4))), "direct"),
    (ctx.MkLe(u[2], ctx.MkAdd(u[0], ctx.MkInt(3))), "limit")
};
string[] exercice2 = null; // TODO : remplacer par les étiquettes d'un MUS de ce modèle
Console.WriteLine(exercice2 == null ? "Exercice 2 à compléter" : $"MUS valide : {IsMus(ex2Constraints, exercice2)}");


Exercice 2 à compléter


## 3. Diagnostic et réparation
Retirer une contrainte d'un MUS rend ce MUS satisfiable, mais un conflit indépendant peut subsister dans le système entier.

### Exercice 3 — Tester une réparation
Choisissez une étiquette à supprimer et vérifiez toutes les contraintes restantes. Le garde refuse une étiquette inconnue et une réparation qui laisse le système UNSAT.

In [6]:
bool Repairs(List<(BoolExpr formula, string label)> xs, string label) => label != null && xs.Count(x => x.label == label) == 1 && Check(xs.Where(x => x.label != label).ToList()) == Status.SATISFIABLE;
string exercice3 = null; // TODO : une étiquette dont le retrait répare le système entier
Console.WriteLine(exercice3 == null ? "Exercice 3 à compléter" : $"Réparation valide : {Repairs(edges, exercice3)}");


Exercice 3 à compléter


## Exercices
Les trois exercices répartis ci-dessus portent sur l'irréductibilité, les chemins transitifs et la réparation globale.

## À retenir
| Objet | Garantie |
|---|---|
| Core | Sous-ensemble insatisfiable |
| MUS | Sous-ensemble insatisfiable irréductible |
| Réparation | Doit être vérifiée sur tout le système restant |

## Vérifiez votre compréhension
Pourquoi un MUS n'est-il pas forcément de cardinal minimum ?
<details><summary>Réponse</summary>Un autre MUS incomparable peut être plus petit.</details>

## Pour aller plus loin
Poursuivez avec [Z3-14 - Bit-vectors](Z3-14-BitVectors-Overflow-Python.ipynb).